# Pipeline de traitement des donnees OpenFoodFacts avec Polars (Par Echantillon)

Ce notebook presente une methode de lecture et de traitement **par echantillon (batch)** du gros fichier CSV de OpenFoodFacts (~12 Go).

Il produit **deux fichiers CSV** :
1. `produits_sans_nutriscore.csv` – Produits francais **SANS** Nutri-Score, tres fournis en nutriments *(cibles de prediction)*
2. `produits_avec_nutriscore.csv` – Produits francais **AVEC** Nutri-Score etabli et toutes les features nutritionnelles presentes *(donnees d'entrainement du Random Forest)*

In [ ]:
import polars as pl
import os

## Configuration
Parametres communs aux deux pipelines.

In [ ]:
FILE_PATH = "data_brut.csv"
BATCH_SIZE = 100_000

# Features nutritionnelles requises pour le Random Forest
FEATURES_RF = [
    'energy_100g',
    'sugars_100g',
    'saturated-fat_100g',
    'salt_100g',
    'fiber_100g',
    'proteins_100g',
    'fruits-vegetables-legumes_100g',
]

COLONNES_NUTRITIONNELLES = '^.*_100g$'

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"Fichier introuvable : {FILE_PATH}")
print(f"Fichier source : {FILE_PATH}")

## Pipeline 1 — Produits francais SANS Nutri-Score
*(cibles de prediction pour le Random Forest)*

Criteres de selection :
- Vendu en France (`countries_en` contient `france`)
- `nutriscore_score` est **null** (pas encore note)
- Au moins **15 colonnes non nulles** (produit suffisamment documente)

In [ ]:
destination_sans = "produits_sans_nutriscore.csv"

reader = pl.read_csv_batched(
    FILE_PATH,
    separator='\t',
    has_header=True,
    ignore_errors=True,
    infer_schema_length=10000,
    low_memory=True,
    quote_char=None,
    batch_size=BATCH_SIZE
)

batches_sans = []
i = 1
print("Debut du Pipeline 1 (sans Nutri-Score)...\n")

while True:
    batches = reader.next_batches(1)
    if not batches:
        break
    df_batch = batches[0]

    df_batch = (
        df_batch
        .filter(pl.col('countries_en').str.to_lowercase().str.contains('france'))
        .select([
            pl.col('code'),
            pl.col('product_name'),
            pl.col('nova_group'),
            pl.col('additives_n'),
            pl.col(COLONNES_NUTRITIONNELLES),
            pl.col('nutriscore_score'),
            pl.col('nutriscore_grade'),
            pl.col('environmental_score_score'),
            pl.col('environmental_score_grade')
        ])
        .filter(pl.col('nutriscore_score').is_null())
        .with_columns(
            pl.sum_horizontal(pl.all().is_not_null()).alias('donnees_fournies_count')
        )
        .filter(pl.col('donnees_fournies_count') >= 15)
    )

    batches_sans.append(df_batch)
    print(f"  Echantillon {i} : {len(df_batch)} produits conserves")
    i += 1

df_sans = pl.concat(batches_sans).sort('donnees_fournies_count', descending=True)
df_sans.write_csv(destination_sans)
print(f"\nFichier sauvegarde : {destination_sans}")
print(f"Dimensions : {df_sans.shape}")
df_sans.head()

## Pipeline 2 — Produits francais AVEC Nutri-Score (donnees d'entrainement)
*(reference pour le Random Forest)*

Criteres de selection :
- Vendu en France (`countries_en` contient `france`)
- `nutriscore_grade` est **une lettre valide** (a / b / c / d / e)
- Toutes les **features nutritionnelles cles du Random Forest** sont presentes (pas de null)

In [ ]:
destination_avec = "produits_avec_nutriscore.csv"

reader2 = pl.read_csv_batched(
    FILE_PATH,
    separator='\t',
    has_header=True,
    ignore_errors=True,
    infer_schema_length=10000,
    low_memory=True,
    quote_char=None,
    batch_size=BATCH_SIZE
)

GRADES_VALIDES = ['a', 'b', 'c', 'd', 'e']
batches_avec = []
j = 1
print("Debut du Pipeline 2 (avec Nutri-Score)...\n")

while True:
    batches = reader2.next_batches(1)
    if not batches:
        break
    df_batch = batches[0]

    df_batch = (
        df_batch
        .filter(pl.col('countries_en').str.to_lowercase().str.contains('france'))
        .select([
            pl.col('code'),
            pl.col('product_name'),
            pl.col('nova_group'),
            pl.col('additives_n'),
            pl.col(COLONNES_NUTRITIONNELLES),
            pl.col('nutriscore_score'),
            pl.col('nutriscore_grade'),
            pl.col('environmental_score_score'),
            pl.col('environmental_score_grade')
        ])
        # Nutri-score valide (a/b/c/d/e)
        .filter(
            pl.col('nutriscore_grade').str.to_lowercase().is_in(GRADES_VALIDES)
        )
        # Toutes les features du RF doivent etre presentes
        .filter(
            pl.all_horizontal(
                [pl.col(f).is_not_null() for f in FEATURES_RF if f in df_batch.columns]
            )
        )
    )

    batches_avec.append(df_batch)
    print(f"  Echantillon {j} : {len(df_batch)} produits conserves")
    j += 1

df_avec = pl.concat(batches_avec)
df_avec.write_csv(destination_avec)
print(f"\nFichier sauvegarde : {destination_avec}")
print(f"Dimensions : {df_avec.shape}")
print("\nDistribution des grades :")
print(df_avec['nutriscore_grade'].value_counts())
df_avec.head()